# **Wan2.1-14b Quantized GGUF Models for Text to Video**

# 🧠 Overview of the Notebook

This Jupyter Notebook is a tool for **generating videos and images from text descriptions**.  
It is divided into two main parts:

1. 🎬 **Text-to-Video Generation** — The core functionality that uses the *Wan2.1-14B* model to create short video clips based on user prompts.  
2. 🔊 **Text-to-Audio Generation** — A smaller demonstration that converts text into speech using a Hugging Face model.

- You can use the free T4 GPU to run this. For faster video generation, use higher GPUs.
- Generating a video with 32 frames at 512 by 910 resolution can take up to 28 minutes on the T4 GPU using the Q5 model.
- It's possible to generate up to 24 frames at 480 by 832 resolution for free using the Q6 model. The generation takes about 20 minutes.
- Set `frames` to 1 to generate an image. You can generate high quality images with 720 by 1280 resolution using either the Q5 or Q6 gguf models. The generation takes less than 7 minutes.

## 🎥 Part 1: Text-to-Video Generation

This is the main section of the notebook. It provides a full workflow to:

- Set up the environment  
- Download and load the required models  
- Generate videos or images based on a text prompt  

### ⚙️ Technology Used

| Component | Description |
|------------|--------------|
| **Model** | `Wan2.1-14B` — A large (14B parameters) text-to-video model |
| **Quantization (GGUF)** | The model is used in GGUF format for memory efficiency, enabling it to run on GPUs like Google Colab’s free T4 |
| **Backend** | Uses **ComfyUI**, a modular node-based system for generative pipelines |
| **Framework** | Clones ComfyUI + a custom `ComfyUI_GGUF` node to integrate quantized models |

---

### 🧩 Core Components

- **UNet (GGUF Model):** The heart of the model that generates video frames in a compressed latent space  
- **Text Encoder (UMT5):** Converts your text prompt into embeddings understood by the UNet  
- **VAE (Variational Autoencoder):** Decodes latent frames into final video or image sequences  

---

### 🧱 Code Breakdown

#### **Cell 1: Prepare Environment**

This cell handles setup and dependencies:

- **Install Libraries:** `torch`, `diffusers`, `xformers`, `av`, etc.  
- **Clone Repositories:** Downloads **ComfyUI** and the **ComfyUI_GGUF** extension  
- **Download Models:** Uses `aria2c` to fetch:
  - The **GGUF UNet model**
  - The **Text Encoder (UMT5)**
  - The **VAE decoder**

> 💡 You can choose between **Q5** and **Q6** quantized models via the `useQ6` flag.  
> Q5 runs faster; Q6 provides slightly better quality.

- **Define Functions:**  
  - `generate_video()` — orchestrates the full text-to-video generation  
  - Helper functions: saving to `.mp4`, `.webm`, or `.png`

In [ ]:
# @title Prepare Environment
!pip install torch==2.6.0 torchvision==0.21.0
%cd /content

!pip install -q torchsde einops diffusers accelerate xformers==0.0.29.post2
!pip install av
!git clone https://github.com/Isi-dev/ComfyUI
%cd /content/ComfyUI/custom_nodes
!git clone https://github.com/Isi-dev/ComfyUI_GGUF.git
%cd /content/ComfyUI/custom_nodes/ComfyUI_GGUF
!pip install -r requirements.txt
%cd /content/ComfyUI
!apt -y install -qq aria2 ffmpeg

useQ6 = False # @param {"type":"boolean"}

if useQ6:
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/wan2.1-t2v-14b-Q6_K.gguf -d /content/ComfyUI/models/unet -o wan2.1-t2v-14b-Q6_K.gguf
else:
    !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/wan2.1-t2v-14b-Q5_0.gguf -d /content/ComfyUI/models/unet -o wan2.1-t2v-14b-Q5_0.gguf

!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors -d /content/ComfyUI/models/text_encoders -o umt5_xxl_fp8_e4m3fn_scaled.safetensors
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors -d /content/ComfyUI/models/vae -o wan_2.1_vae.safetensors

import torch
import numpy as np
from PIL import Image
import gc
import sys
import random
import os
import imageio
import subprocess
from google.colab import files
from IPython.display import display, HTML, Image as IPImage
sys.path.insert(0, '/content/ComfyUI')

from comfy import model_management

from nodes import (
    CheckpointLoaderSimple,
    CLIPLoader,
    CLIPTextEncode,
    VAEDecode,
    VAELoader,
    KSampler,
    UNETLoader
)

from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
from comfy_extras.nodes_model_advanced import ModelSamplingSD3
from comfy_extras.nodes_hunyuan import EmptyHunyuanLatentVideo
from comfy_extras.nodes_images import SaveAnimatedWEBP
from comfy_extras.nodes_video import SaveWEBM

# unet_loader = UNETLoader()
unet_loader = UnetLoaderGGUF()
# model_sampling = ModelSamplingSD3()
clip_loader = CLIPLoader()
clip_encode_positive = CLIPTextEncode()
clip_encode_negative = CLIPTextEncode()
vae_loader = VAELoader()
empty_latent_video = EmptyHunyuanLatentVideo()
ksampler = KSampler()
vae_decode = VAEDecode()
save_webp = SaveAnimatedWEBP()
save_webm = SaveWEBM()

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    for obj in list(globals().values()):
        if torch.is_tensor(obj) or (hasattr(obj, "data") and torch.is_tensor(obj.data)):
            del obj
    gc.collect()

def save_as_mp4(images, filename_prefix, fps, output_dir="/content/ComfyUI/output"):
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.mp4"

    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]

    with imageio.get_writer(output_path, fps=fps) as writer:
        for frame in frames:
            writer.append_data(frame)

    return output_path

def save_as_webp(images, filename_prefix, fps, quality=90, lossless=False, method=4, output_dir="/content/ComfyUI/output"):
    """Save images as animated WEBP using imageio."""
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webp"


    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]


    kwargs = {
        'fps': int(fps),
        'quality': int(quality),
        'lossless': bool(lossless),
        'method': int(method)
    }

    with imageio.get_writer(
        output_path,
        format='WEBP',
        mode='I',
        **kwargs
    ) as writer:
        for frame in frames:
            writer.append_data(frame)

    return output_path

def save_as_webm(images, filename_prefix, fps, codec="vp9", quality=32, output_dir="/content/ComfyUI/output"):
    """Save images as WEBM using imageio."""
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.webm"


    frames = [(img.cpu().numpy() * 255).astype(np.uint8) for img in images]


    kwargs = {
        'fps': int(fps),
        'quality': int(quality),
        'codec': str(codec),
        'output_params': ['-crf', str(int(quality))]
    }

    with imageio.get_writer(
        output_path,
        format='FFMPEG',
        mode='I',
        **kwargs
    ) as writer:
        for frame in frames:
            writer.append_data(frame)

    return output_path

def save_as_image(image, filename_prefix, output_dir="/content/ComfyUI/output"):
    """Save single frame as PNG image."""
    os.makedirs(output_dir, exist_ok=True)
    output_path = f"{output_dir}/{filename_prefix}.png"

    frame = (image.cpu().numpy() * 255).astype(np.uint8)

    Image.fromarray(frame).save(output_path)

    return output_path

def generate_video(
    positive_prompt: str = "a fox moving quickly in a beautiful winter scenery nature trees mountains daytime tracking camera",
    negative_prompt: str = "Tones are bright, overexposed, static, details are unclear, text, style, artwork, painting, screen, still, overall gray, poor quality, low quality, JPEG compression residue, dirty, gray, missing, more Some fingers, poorly drawn hand part, poorly drawn face part, distorted shape, distorted body shape, distorted body shape, finger fusion, still non-moving picture, messed up background, three-string legs, too many people in the background, backward movement",
    width: int = 832,
    height: int = 480,
    seed: int = 82628696717253,
    steps: int = 30,
    cfg_scale: float = 1.0,
    sampler_name: str = "uni_pc",
    scheduler: str = "simple",
    frames: int = 33,
    fps: int = 16,
    output_format: str = "mp4"
):

    with torch.inference_mode():
        print("Loading Text_Encoder...")
        clip = clip_loader.load_clip("umt5_xxl_fp8_e4m3fn_scaled.safetensors", "wan", "default")[0]

        positive = clip_encode_positive.encode(clip, positive_prompt)[0]
        negative = clip_encode_negative.encode(clip, negative_prompt)[0]

        del clip
        torch.cuda.empty_cache()
        gc.collect()

        empty_latent = empty_latent_video.generate(width, height, frames, 1)[0]

        print("Loading Unet Model...")
        if useQ6:
            model = unet_loader.load_unet("wan2.1-t2v-14b-Q6_K.gguf")[0]
        else:
            model = unet_loader.load_unet("wan2.1-t2v-14b-Q5_0.gguf")[0]
        # model = model_sampling.patch(model, 8)[0]

        print("Generating video...")
        sampled = ksampler.sample(
            model=model,
            seed=seed,
            steps=steps,
            cfg=cfg_scale,
            sampler_name=sampler_name,
            scheduler=scheduler,
            positive=positive,
            negative=negative,
            latent_image=empty_latent
        )[0]

        del model
        torch.cuda.empty_cache()
        gc.collect()

        print("Loading VAE...")
        vae = vae_loader.load_vae("wan_2.1_vae.safetensors")[0]

        try:
            print("Decoding latents...")
            decoded = vae_decode.decode(vae, sampled)[0]

            del vae
            torch.cuda.empty_cache()
            gc.collect()

            output_path = ""
            if frames == 1:
                print("Single frame detected - saving as PNG image...")
                output_path = save_as_image(decoded[0], "ComfyUI")
                # print(f"Image saved as PNG: {output_path}")

                display(IPImage(filename=output_path))
            else:
                if output_format.lower() == "webm":
                    print("Saving as WEBM...")
                    output_path = save_as_webm(
                        decoded,
                        "ComfyUI",
                        fps=fps,
                        codec="vp9",
                        quality=10
                    )
                elif output_format.lower() == "mp4":
                    print("Saving as MP4...")
                    output_path = save_as_mp4(decoded, "ComfyUI", fps)
                else:
                    raise ValueError(f"Unsupported output format: {output_format}")

                # print(f"Video saved as {output_format.upper()}: {output_path}")

                display_video(output_path)

        except Exception as e:
            print(f"Error during decoding/saving: {str(e)}")
            raise
        finally:
            clear_memory()

def display_video(video_path):
    from IPython.display import HTML
    from base64 import b64encode

    video_data = open(video_path,'rb').read()

    # Determine MIME type based on file extension
    if video_path.lower().endswith('.mp4'):
        mime_type = "video/mp4"
    elif video_path.lower().endswith('.webm'):
        mime_type = "video/webm"
    elif video_path.lower().endswith('.webp'):
        mime_type = "image/webp"
    else:
        mime_type = "video/mp4"  # default

    data_url = f"data:{mime_type};base64," + b64encode(video_data).decode()

    display(HTML(f"""
    <video width=512 controls autoplay loop>
        <source src="{data_url}" type="{mime_type}">
    </video>
    """))

print("✅ Environment Setup Complete!")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 882.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 579.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/content
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 17.3 MB/s eta 0:00:00
Cloning into 'ComfyUI'...
remote: Enumerating objects: 21665, done.
remote: Total 21665 (delta 0), reused 0 (delta 0), pack-reused 21665 (from 1)
Receiving objects: 100% (21665/21665), 65.81 MiB | 21.73 MiB/s, done.
Resolving deltas: 100% (14420/14420), done.
/content/ComfyUI/custom_nodes
Cloning into 'ComfyUI_GGUF'...
remote: Enumerating objects: 609, done.
remote: Counting objects: 100% (297/297), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 609 (delta 264), reused 158 (delta 158), pack-reused 312 (from 2)
Receiving objects: 100% (609/609), 167.35 KiB | 1.29 MiB/s, done.
Resolving deltas: 100% (390/390), done.
/content/ComfyUI/custom_nodes/ComfyUI_GGUF
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 5.3 MB/s 

    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.8.0+cu126)
    Python  3.12.8 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


✅ Environment Setup Complete!


#### **Cell 2: Generate Video/Image**

Where you trigger the generation process.

**Adjustable Parameters:**

| Parameter | Description |
|------------|--------------|
| `positive_prompt` | What you want in the video |
| `negative_prompt` | What to avoid |
| `width`, `height` | Resolution of output |
| `frames` | Number of frames (1 → image only) |
| `fps` | Frames per second |
| `steps` | Diffusion steps controlling detail and smoothness |


In [1]:
# @title Generate Video/Image

positive_prompt = "a fox walking on bright moon" # @param {"type":"string"}
negative_prompt = "Tones are bright, overexposed, static, details are unclear, text, style, artwork, painting, screen, still, overall gray, poor quality, low quality, JPEG compression residue, dirty, gray, missing, more Some fingers, poorly drawn hand part, poorly drawn face part, distorted shape, distorted body shape, distorted body shape, finger fusion, still non-moving picture, messed up background, three-string legs, too many people in the background, backward movement" # @param {"type":"string"}
width = 512 # @param {"type":"number"}
height = 288 # @param {"type":"number"}
seed = 82628696717258 # @param {"type":"integer"}
steps = 10 # @param {"type":"integer", "min":1, "max":100}
cfg_scale = 1.2 # @param {"type":"number", "min":1, "max":20}
sampler_name = "uni_pc" # @param ["uni_pc", "euler", "dpmpp_2m", "ddim", "lms"]
scheduler = "simple" # @param ["simple", "normal", "karras", "exponential"]
frames = 25 # @param {"type":"integer", "min":1, "max":120}
fps = 5 # @param {"type":"integer", "min":1, "max":60}
output_format = "mp4" # @param ["mp4", "webm"]

# with torch.inference_mode():
generate_video(
    positive_prompt=positive_prompt,
    negative_prompt=negative_prompt,
    width=width,
    height=height,
    seed=seed,
    steps=steps,
    cfg_scale=cfg_scale,
    sampler_name=sampler_name,
    scheduler=scheduler,
    frames=frames,
    fps=fps,
    output_format=output_format
)
clear_memory()

NameError: name 'generate_video' is not defined

# Demo code on generating audio from text using Hugging Face and LLM

## 🔈 Part 2: Text-to-Audio (Text-to-Speech) Generation

A standalone demo that converts text into audio using **Hugging Face Transformers**.

### ⚙️ Technology Used

| Component | Description |
|------------|--------------|
| **Library** | `transformers` |
| **Model** | `facebook/mms-tts-eng` — from Facebook’s *Massively Multilingual Speech (MMS)* project |

---

### 🧱 Code Breakdown

#### **Cell 1: Load Model**
- Imports `VitsModel` and `AutoTokenizer`
- Loads pre-trained English TTS model and tokenizer

In [1]:
from transformers import VitsModel, AutoTokenizer
import torch

model = VitsModel.from_pretrained("facebook/mms-tts-eng")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-eng")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/413 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

#### **Cell 2: Generate Waveform**
- Defines sample text  
- Tokenizes it into input for the model  
- Generates an **audio waveform** output

In [11]:

text = "a fox running across snowy mountains, cinematic camera tracking, natural light"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    output = model(**inputs).waveform

#### **Cell 3: Play Audio**
- Uses `IPython.display.Audio` to embed an audio player  
- Allows direct playback of the synthesized voice

In [12]:
from IPython.display import Audio

Audio(output.numpy(), rate=model.config.sampling_rate)
